# Predicción de Lluvia en Australia — PCA + Regresión Logística

**Asignatura:** Inteligencia Artificial I — Actividad 3  
**Algoritmo:** PCA como preprocesamiento para clasificación  
**Dataset:** Rain in Australia — Kaggle (jsphyg/weather-dataset-rattle-package)  

### Pipeline
```
Datos crudos → EDA → Limpieza → StandardScaler → PCA → LogisticRegression → Serialización
```

### Descarga del dataset
1. Entra a https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package
2. Haz clic en **Download** (requiere cuenta Kaggle gratuita)
3. Descomprime y coloca `weatherAUS.csv` en la carpeta `data/raw/`

**O con la API de Kaggle:**
```bash
pip install kaggle
kaggle datasets download -d jsphyg/weather-dataset-rattle-package -p data/raw/ --unzip
```

## 0. Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Librerías importadas correctamente ✓')

## 1. Carga de Datos

Fuente: [Kaggle — Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package)

El dataset contiene 10 años de observaciones meteorológicas diarias de 49 estaciones en Australia.  
El objetivo es predecir si lloverá al día siguiente (**RainTomorrow**).

In [ ]:
# Carga del dataset
DATA_PATH = '../data/raw/weatherAUS.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'\nColumnas: {list(df.columns)}')
df.head()

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Información general
print('=== Información del Dataset ===')
print(df.dtypes.to_string())

# Valores nulos
print('\n=== Valores Nulos (top 10) ===')
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls > 0].to_string())

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df['RainTomorrow'].value_counts()
axes[0].bar(['No lluvia', 'Lluvia'], target_counts.values, color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Distribución de RainTomorrow', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Cantidad de registros')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['No lluvia', 'Lluvia'],
            colors=['#3498db', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
os.makedirs('../data/processed', exist_ok=True)
plt.savefig('../data/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuciones de variables numéricas clave
NUM_FEATURES = ['MinTemp', 'MaxTemp', 'Rainfall', 'Humidity9am', 'Humidity3pm',
                'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm', 'WindGustSpeed']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, feat in enumerate(NUM_FEATURES):
    no_rain = df[df['RainTomorrow'] == 'No'][feat].dropna()
    rain    = df[df['RainTomorrow'] == 'Yes'][feat].dropna()
    axes[i].hist(no_rain, alpha=0.6, label='No lluvia', color='#3498db', bins=30)
    axes[i].hist(rain,    alpha=0.6, label='Lluvia',    color='#e74c3c', bins=30)
    axes[i].set_title(feat, fontsize=11)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribución de Variables por Clase', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de correlación — EVIDENCIA de por qué PCA tiene sentido aquí
NUMERICAL_ALL = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
                 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm',
                 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm',
                 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']

fig, ax = plt.subplots(figsize=(14, 12))
corr = df[NUMERICAL_ALL].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Matriz de Correlación — Variables Numéricas\n'
             '(Notar alta correlación entre mediciones de 9am y 3pm → PCA ideal)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Correlaciones destacadas:')
print(f'  Temp9am vs Temp3pm:      {corr.loc["Temp9am","Temp3pm"]:.3f}')
print(f'  Pressure9am vs P3pm:     {corr.loc["Pressure9am","Pressure3pm"]:.3f}')
print(f'  Humidity9am vs H3pm:     {corr.loc["Humidity9am","Humidity3pm"]:.3f}')
print(f'  MinTemp vs MaxTemp:      {corr.loc["MinTemp","MaxTemp"]:.3f}')
print('→ Alta correlación = redundancia = PCA reduce sin perder información relevante')

## 3. Preprocesamiento

In [ ]:
# Features numéricas (16) + RainToday binaria (1) = 17 features totales
FEATURES = NUMERICAL_ALL + ['RainToday']
TARGET = 'RainTomorrow'

# Seleccionar y copiar
data = df[FEATURES + [TARGET]].copy()

# Eliminar filas sin target
data = data.dropna(subset=[TARGET])
print(f'Registros después de limpiar target nulo: {len(data):,}')

# Codificar binarias
data['RainToday']    = data['RainToday'].map({'Yes': 1, 'No': 0})
data[TARGET]         = data[TARGET].map({'Yes': 1, 'No': 0})

# Rellenar nulos con la mediana de cada columna
for col in NUMERICAL_ALL:
    data[col].fillna(data[col].median(), inplace=True)
data['RainToday'].fillna(0, inplace=True)

print(f'Valores nulos restantes: {data.isnull().sum().sum()}')

X = data[FEATURES]
y = data[TARGET]

# Guardar rangos para la app
feature_ranges = {
    col: {'min': float(X[col].min()), 'max': float(X[col].max()),
          'mean': float(X[col].mean()), 'median': float(X[col].median())}
    for col in FEATURES
}

print(f'\nX shape: {X.shape}')
print(f'Distribución target: No={( y==0).sum():,} | Sí={( y==1).sum():,}')

In [ ]:
# División train/test estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]:,} muestras | Test: {X_test.shape[0]:,} muestras')

# Estandarizar — OBLIGATORIO antes de PCA
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Estandarizado: media={X_train_s.mean():.4f}, std={X_train_s.std():.4f}')

## 4. Aplicación de PCA

**¿Por qué PCA aquí?**  
Las 17 variables meteorológicas presentan alta correlación:
- Temp9am y Temp3pm (r ≈ 0.98)
- Pressure9am y Pressure3pm (r ≈ 0.96)
- Humidity9am y Humidity3pm (r ≈ 0.70)
- MinTemp, MaxTemp y ambas temperaturas diurnas (r > 0.85)

PCA colapsa estas correlaciones en componentes ortogonales independientes, eliminando la redundancia sin perder la varianza que explica si lloverá mañana.

In [ ]:
# Análisis de varianza explicada
pca_full = PCA()
pca_full.fit(X_train_s)

ev = pca_full.explained_variance_ratio_
cum = np.cumsum(ev)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(ev)+1), ev*100, color='#3498db', edgecolor='black')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada (%)')
axes[0].set_title('Varianza por Componente', fontweight='bold')

axes[1].plot(range(1, len(cum)+1), cum*100, 'bo-', linewidth=2)
axes[1].axhline(y=95, color='r', linestyle='--', label='95% varianza')
axes[1].axhline(y=99, color='orange', linestyle='--', label='99% varianza')
axes[1].fill_between(range(1, len(cum)+1), cum*100, alpha=0.15)
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada (%)')
axes[1].set_title('Varianza Acumulada', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../data/processed/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

n_95 = np.argmax(cum >= 0.95) + 1
n_99 = np.argmax(cum >= 0.99) + 1
print(f'Componentes para 95% de varianza: {n_95}')
print(f'Componentes para 99% de varianza: {n_99}')
print(f'Reducción: {X.shape[1]} → {n_95} dimensiones ({(1-n_95/X.shape[1])*100:.1f}% menos)')

In [ ]:
# Aplicar PCA con 95% de varianza
pca = PCA(n_components=0.95, random_state=42)
X_train_p = pca.fit_transform(X_train_s)
X_test_p  = pca.transform(X_test_s)

print(f'Dimensiones originales:  {X_train_s.shape[1]} features')
print(f'Dimensiones reducidas:   {pca.n_components_} componentes')
print(f'Varianza preservada:     {sum(pca.explained_variance_ratio_)*100:.2f}%')

In [ ]:
# Visualización: las 2 primeras componentes principales
# Muestra solo 2000 puntos para agilizar
idx = np.random.choice(len(X_train_p), min(2000, len(X_train_p)), replace=False)

fig, ax = plt.subplots(figsize=(10, 7))
for cls, color, label in [(0, '#3498db', 'No lluvia'), (1, '#e74c3c', 'Lluvia')]:
    mask = y_train.values[idx] == cls
    ax.scatter(X_train_p[idx][mask, 0], X_train_p[idx][mask, 1],
               c=color, label=label, alpha=0.5, edgecolors='none', s=25)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)', fontsize=12)
ax.set_title('Proyección en los 2 Primeros Componentes Principales', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Entrenamiento

In [ ]:
# Regresión Logística con pesos balanceados (hay más días sin lluvia que con lluvia)
from sklearn.pipeline import Pipeline

classifier = LogisticRegression(random_state=42, max_iter=2000,
                                 class_weight='balanced', solver='lbfgs')
classifier.fit(X_train_p, y_train)

# Validación cruzada
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=0.95, random_state=42)),
    ('clf',    LogisticRegression(random_state=42, max_iter=2000, class_weight='balanced'))
])

cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='f1')
print(f'Validación cruzada F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('Modelo entrenado ✓')

## 6. Evaluación

In [ ]:
y_pred = classifier.predict(X_test_p)
y_prob = classifier.predict_proba(X_test_p)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('='*45)
print('       MÉTRICAS DE EVALUACIÓN')
print('='*45)
print(f'  Accuracy:   {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision:  {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall:     {rec:.4f}  ({rec*100:.2f}%)')
print(f'  F1-Score:   {f1:.4f}  ({f1*100:.2f}%)')
print(f'  ROC-AUC:    {auc:.4f}')
print('='*45)
print(f'\n{classification_report(y_test, y_pred, target_names=["No lluvia", "Lluvia"])}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No lluvia', 'Lluvia'],
            yticklabels=['No lluvia', 'Lluvia'])
axes[0].set_title('Matriz de Confusión', fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'r--')
axes[1].fill_between(fpr, tpr, alpha=0.2)
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC', fontweight='bold')
axes[1].legend()
axes[1].grid(True)

# Barras de métricas
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
values  = [acc, prec, rec, f1, auc]
colors_b = ['#2ecc71' if v >= 0.7 else '#e74c3c' for v in values]
bars = axes[2].barh(metrics, values, color=colors_b, edgecolor='black')
axes[2].set_xlim(0, 1.1)
axes[2].axvline(x=0.7, color='red', linestyle='--', alpha=0.7, label='Mínimo 70%')
for bar, v in zip(bars, values):
    axes[2].text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontweight='bold')
axes[2].set_title('Resumen de Métricas', fontweight='bold')
axes[2].legend()

plt.tight_layout()
plt.savefig('../data/processed/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Serialización

In [ ]:
os.makedirs('../models', exist_ok=True)

joblib.dump(scaler,         '../models/scaler.pkl')
joblib.dump(pca,            '../models/pca.pkl')
joblib.dump(classifier,     '../models/classifier.pkl')
joblib.dump(feature_ranges, '../models/feature_ranges.pkl')
joblib.dump(FEATURES,       '../models/feature_names.pkl')

# Verificación
s2  = joblib.load('../models/scaler.pkl')
p2  = joblib.load('../models/pca.pkl')
c2  = joblib.load('../models/classifier.pkl')
sample = X_test.iloc[[0]]
pred_v = c2.predict(p2.transform(s2.transform(sample)))[0]
real_v = int(y_test.iloc[0])
print(f'Verificación: predicho={pred_v} real={real_v} → {"✓ OK" if pred_v==real_v else "✗ ERROR"}')

print('\nResumen final:')
print(f'  Features:           {len(FEATURES)}')
print(f'  Componentes PCA:    {pca.n_components_}')
print(f'  Varianza explicada: {sum(pca.explained_variance_ratio_)*100:.1f}%')
print(f'  Accuracy test:      {acc*100:.2f}%')
print(f'  F1-Score test:      {f1*100:.2f}%')
print(f'  ROC-AUC:            {auc:.4f}')
print('\nSerialización completada ✓')